In [4]:
import os
from datetime import datetime
from typing import Annotated, Literal, TypedDict

from langchain_core.messages import HumanMessage, SystemMessage, ToolMessage
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableParallel
from langchain_core.tools import tool
from langchain_ollama import ChatOllama

from pydantic import BaseModel, Field

from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langgraph.checkpoint.memory import InMemorySaver


MODELO = "qwen3:4b"

## 1) Introducción a LangChain


### Basic Chat 

Un ChatModel recibe una lista de mensaje. Hay tres
tipos básicos: 
- SystemMessage (instrucciones)
- HumanMessage (usuario)
- AIMessage (respuesta del modelo)

Esta abstracción es común a Anthropic, OpenAI, Google, etc., así que cambiar de LLM es trivial.

In [ ]:
llm = ChatOllama(
    model=MODELO, 
    temperature=0, 
    num_thread=os.cpu_count(), 
    num_gpu=999, # aplica solo a silicon mac
    model_kwargs={"think": False}
)
mensajes = [
    SystemMessage("Responde siempre en una sola frase."),
    HumanMessage("¿Qué es un tensor en deep learning?"),
]
respuesta = llm.invoke(mensajes)
print(respuesta.text)

Se puede realizar la llamada sin un mensaje de systema, el LLM utilizaría el SystemMessage por defecto: you are a helpful asistant

In [18]:
mensajes = [
    HumanMessage("¿Qué es un tensor en deep learning?"),
]
respuesta = llm.invoke(mensajes)
print(respuesta.text)

En deep learning, un tensor es una estructura de datos multidimensional que generaliza a vectores y matrices. Se utilizan para representar datos como imágenes (en formato 3D: altura, anchura y canales) o textos (en formato 2D: longitud de secuencia y tamaño del vocabulario), así como los parámetros de los modelos neuronales. Frameworks como TensorFlow y PyTorch manejan tensores para realizar operaciones matemáticas eficientemente en hardware como GPUs.


### Aplicación de plantillas

Las plantillas facilitan el uso en bucles de un LLM

In [20]:
plantilla = ChatPromptTemplate.from_messages([
    ("system", "Eres un experto en {dominio}. Sé conciso."),
    ("human", "{pregunta}"),
])
mensajes = plantilla.invoke({
    "dominio": "estadística bayesiana",
    "pregunta": "¿Qué es un prior conjugado?",
})
print(mensajes)
respuesta = llm.invoke(mensajes)
print(respuesta.text)

messages=[SystemMessage(content='Eres un experto en estadística bayesiana. Sé conciso.', additional_kwargs={}, response_metadata={}), HumanMessage(content='¿Qué es un prior conjugado?', additional_kwargs={}, response_metadata={})]
Un prior conjugado es una distribución de probabilidad que, al combinarse con la función de verosimilitud, genera un posterior perteneciente a la misma familia que el prior. Esto permite calcular analíticamente el posterior.


### Forzar salidas estructuradas

In [22]:
class Resena(BaseModel):
    """Reseña estructurada de un paper."""
    titulo: str = Field(description="Título del paper")
    aporte_principal: str = Field(description="Contribución central en una frase")
    confianza: Literal["alta", "media", "baja"]

llm = ChatOllama(
    model=MODELO, 
    temperature=0, 
    num_thread=os.cpu_count(), 
    num_gpu=999, # aplica solo a silicon mac
    model_kwargs={"think": False}
).with_structured_output(Resena)

salida = llm.invoke("Resume en una oración corta 'Attention Is All You Need' (Vaswani et al., 2017).")
print(type(salida).__name__, "->", salida)

Resena -> titulo='Attention Is All You Need' aporte_principal='Introducción del modelo Transformer que utiliza exclusivamente mecanismos de atención para procesar secuencias, eliminando la necesidad de redes recurrentes' confianza='alta'


### Uso de herramientas

In [ ]:
@tool
def multiplicar(a: float, b: float) -> float:
    return a * b


@tool
def hora_actual() -> str:
    return datetime.now().isoformat(timespec="seconds")



llm = ChatOllama(
    model=MODELO, 
    temperature=0, 
    num_thread=os.cpu_count(), 
    num_gpu=999, # aplica solo a silicon mac
    model_kwargs={"think": False}
)
llm_con_tools = llm.bind_tools([multiplicar, hora_actual])
tools_map = {"multiplicar": multiplicar, "hora_actual": hora_actual}

# Historial inicial
mensajes = [HumanMessage("¿Cuánto es 23.5 por 17? Y dime la hora.")]

# Primera llamada: el modelo pide las tools
respuesta = llm_con_tools.invoke(mensajes)
print(f"{respuesta=}")
mensajes.append(respuesta)  # añadir AIMessage con tool_calls

# Ejecutar cada tool y añadir su resultado al historial
for tc in respuesta.tool_calls:
    tool_fn = tools_map[tc["name"]]
    resultado = tool_fn.invoke(tc["args"])
    print(f"{resultado=}")
    
    mensajes.append(ToolMessage(
        content=str(resultado),
        tool_call_id=tc["id"],   # ← debe coincidir con el id del tool_call
    ))

# Segunda llamada: el modelo genera la respuesta final con los resultados
respuesta_final = llm_con_tools.invoke(mensajes)
print(respuesta_final.content)

respuesta=AIMessage(content='', additional_kwargs={}, response_metadata={'model': 'qwen3:4b', 'created_at': '2026-05-07T01:47:16.640087Z', 'done': True, 'done_reason': 'stop', 'total_duration': 15056581166, 'load_duration': 70907291, 'prompt_eval_count': 213, 'prompt_eval_duration': 117889459, 'eval_count': 446, 'eval_duration': 14758407761, 'logprobs': None, 'model_name': 'qwen3:4b', 'model_provider': 'ollama'}, id='lc_run--019e001d-ec4e-75c0-9214-34359f907071-0', tool_calls=[{'name': 'multiplicar', 'args': {'a': 23.5, 'b': 17}, 'id': '4bd120d4-c096-49a2-8cf8-6cfadd5d5723', 'type': 'tool_call'}, {'name': 'hora_actual', 'args': {}, 'id': 'b1bd1a4b-35a6-45e5-aa07-03698d8517ef', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 213, 'output_tokens': 446, 'total_tokens': 659})
resultado=399.5
resultado='2026-05-06T20:47:16'
23.5 por 17 es 399.5. La hora actual es 2026-05-06T20:47:16.


In [33]:
mensajes

[HumanMessage(content='¿Cuánto es 23.5 por 17? Y dime la hora.', additional_kwargs={}, response_metadata={}),
 AIMessage(content='', additional_kwargs={}, response_metadata={'model': 'qwen3:4b', 'created_at': '2026-05-07T01:47:16.640087Z', 'done': True, 'done_reason': 'stop', 'total_duration': 15056581166, 'load_duration': 70907291, 'prompt_eval_count': 213, 'prompt_eval_duration': 117889459, 'eval_count': 446, 'eval_duration': 14758407761, 'logprobs': None, 'model_name': 'qwen3:4b', 'model_provider': 'ollama'}, id='lc_run--019e001d-ec4e-75c0-9214-34359f907071-0', tool_calls=[{'name': 'multiplicar', 'args': {'a': 23.5, 'b': 17}, 'id': '4bd120d4-c096-49a2-8cf8-6cfadd5d5723', 'type': 'tool_call'}, {'name': 'hora_actual', 'args': {}, 'id': 'b1bd1a4b-35a6-45e5-aa07-03698d8517ef', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 213, 'output_tokens': 446, 'total_tokens': 659}),
 ToolMessage(content='399.5', tool_call_id='4bd120d4-c096-49a2-8cf8-6cfadd5d5723'),
 

In [34]:
for chunk in llm_con_tools.stream(mensajes):
    print(chunk.content, end="", flush=True)
print()


23.5 por 17 es 399.5. La hora actual es 2026-05-06T20:47:16.


### Uso de cadenas

In [ ]:
plantilla = ChatPromptTemplate.from_template("Explica {tema} en una frase.")
llm = ChatOllama(
    model=MODELO, 
    temperature=0, 
    num_thread=os.cpu_count(), 
    num_gpu=999, # aplica solo a silicon mac
    model_kwargs={"think": False}
)
cadena = plantilla | llm | StrOutputParser() # StrOutputParser aplica .content
print(cadena.invoke({"tema": "el teorema de Bayes"}))

El teorema de Bayes describe cómo actualizar la probabilidad de una hipótesis en función de nuevos datos, utilizando la probabilidad previa y la probabilidad de los datos condicionada a dicha hipótesis.


In [36]:
plantilla = ChatPromptTemplate.from_template("Explica {tema} en una frase.")
llm = ChatOllama(
    model=MODELO, 
    temperature=0, 
    num_thread=os.cpu_count(), 
    num_gpu=999, # aplica solo a silicon mac
    model_kwargs={"think": False}
)
cadena = plantilla | llm 
print(cadena.invoke({"tema": "el teorema de Bayes"}))

content='El teorema de Bayes describe cómo actualizar la probabilidad de una hipótesis en función de nuevos datos, utilizando la probabilidad previa y la probabilidad de los datos condicionada a dicha hipótesis.' additional_kwargs={} response_metadata={'model': 'qwen3:4b', 'created_at': '2026-05-07T01:56:38.175494Z', 'done': True, 'done_reason': 'stop', 'total_duration': 88674105875, 'load_duration': 76294625, 'prompt_eval_count': 23, 'prompt_eval_duration': 111293875, 'eval_count': 2473, 'eval_duration': 87860906151, 'logprobs': None, 'model_name': 'qwen3:4b', 'model_provider': 'ollama'} id='lc_run--019e0025-5e3b-7c13-a9ec-0d1596b1b11a-0' tool_calls=[] invalid_tool_calls=[] usage_metadata={'input_tokens': 23, 'output_tokens': 2473, 'total_tokens': 2496}


### Ejecución en paralelo

In [37]:
llm = ChatOllama(
    model=MODELO, 
    temperature=0, 
    num_thread=os.cpu_count(), 
    num_gpu=999, # aplica solo a silicon mac
    model_kwargs={"think": False}
)
capital = ChatPromptTemplate.from_template("Capital de {pais}, solo el nombre.") | llm | StrOutputParser()
moneda  = ChatPromptTemplate.from_template("Moneda de {pais}, solo código ISO.") | llm | StrOutputParser()

paralelo = RunnableParallel(capital=capital, moneda=moneda)
print(paralelo.invoke({"pais": "Japón"}))


{'capital': 'Tokyo', 'moneda': 'JPY'}


## 2) LangGraph

### Grafo simple
Encadenar mediante `|` es perfecto para flujos lineales, para flujos no lineales usaremos LangGraph

In [46]:
class EstadoSimple(TypedDict):
    numero: int
    log: list[str]

# Cada nodo devuelve un diccionario parcial; LangGraph lo fusiona con el estado global. 
def sumar_5(state: EstadoSimple) -> dict:
    return {"numero": state["numero"] + 5, "log": state["log"] + ["sumé 5"]}

def doblar(state: EstadoSimple) -> dict:
    return {"numero": state["numero"] * 2, "log": state["log"] + ["doblé"]}

grafo = StateGraph(EstadoSimple)
grafo.add_node("sumar", sumar_5) # denominar función sumar_5 a str "sumar"
grafo.add_node("doblar", doblar)
grafo.add_edge(START, "sumar") # Iniciar grafo, primer nodo: "sumar"
grafo.add_edge("sumar", "doblar") # Segundo elemento de grafo, primer nodo "sumar" + segundo nodo "doblar"
grafo.add_edge("doblar", END) # Cerrar el grafo tras nodo "doblar"

app = grafo.compile()
print(app.invoke({"numero": 10, "log": []}))

{'numero': 30, 'log': ['sumé 5', 'doblé']}


```
Estado inicial:      {"numero": 10, "log": []}
                              │
                         nodo "sumar"
                    numero = 10 + 5 = 15
                    log = [] + ["sumé 5"]
                              │
                    {"numero": 15, "log": ["sumé 5"]}
                              │
                         nodo "doblar"
                    numero = 15 * 2 = 30
                    log = ["sumé 5"] + ["doblé"]
                              │
                    {"numero": 30, "log": ["sumé 5", "doblé"]}
```

### Grafo condicional

In [49]:
llm = ChatOllama(
    model=MODELO, 
    temperature=0, 
    num_thread=os.cpu_count(), 
    num_gpu=999, # aplica solo a silicon mac
    model_kwargs={"think": False}
)

class EstadoClasificado(TypedDict):
    texto: str
    categoria: str
    salida: str

def clasificar(state: EstadoClasificado) -> dict:
    prompt = (
        "Clasifica este mensaje en UNA palabra exacta entre "
        "{queja, pregunta, elogio}. Mensaje: " + state["texto"]
    )
    return {"categoria": llm.invoke(prompt).content.strip().lower()}

def manejar_queja(state):    return {"salida": "[abro ticket de soporte]"}
def manejar_pregunta(state): return {"salida": "[paso al asistente FAQ]"}
def manejar_elogio(state):   return {"salida": "[envío agradecimiento]"}

def router(state: EstadoClasificado) -> Literal["queja", "pregunta", "elogio"]:
    if "queja" in state["categoria"]:    return "queja"
    if "pregunta" in state["categoria"]: return "pregunta"
    return "elogio"

g = StateGraph(EstadoClasificado)
g.add_node("clasificar", clasificar)
g.add_node("queja", manejar_queja)
g.add_node("pregunta", manejar_pregunta)
g.add_node("elogio", manejar_elogio)
g.add_edge(START, "clasificar")
g.add_conditional_edges("clasificar", router)
for nodo in ("queja", "pregunta", "elogio"):
    g.add_edge(nodo, END)

app = g.compile()
res = app.invoke({"texto": "Llevo tres días esperando mi pedido.",
                    "categoria": "", "salida": ""})
print(res)

{'texto': 'Llevo tres días esperando mi pedido.', 'categoria': 'queja', 'salida': '[abro ticket de soporte]'}


```
Estado inicial:   {"texto": "Llevo tres días...", "categoria": "", "salida": ""}
                                        │
                                  nodo "clasificar"
                         prompt → LLM → .content.strip().lower()
                                categoria = "queja"
                                        │
                         {"texto": "...", "categoria": "queja", "salida": ""}
                                        │
                                    router(state)
                                        │
                  ┌─────────────────────┼─────────────────────┐
          "queja" in categoria?  "pregunta" in categoria?     else
                  │                     │                      │
                  ▼                     ▼                      ▼
          nodo "queja"          nodo "pregunta"         nodo "elogio"
     salida = "[abro          salida = "[paso al      salida = "[envío
     ticket de soporte]"       asistente FAQ]"        agradecimiento]"
                  │                     │                      │
                  └─────────────────────┴──────────────────────┘
                                        │
                                       END

════════════════════ camino tomado en esta ejecución ════════════════════

Estado inicial:   {"texto": "Llevo tres días...", "categoria": "", "salida": ""}
                                        │
                                  nodo "clasificar"
                                categoria = "queja"
                                        │
                         {"texto": "...", "categoria": "queja", "salida": ""}
                                        │
                                    router(state)
                              "queja" in "queja" → True
                                        │
                                  nodo "queja"
                         salida = "[abro ticket de soporte]"
                                        │
                  {"texto": "...", "categoria": "queja",
                   "salida": "[abro ticket de soporte]"}
                                        │
                                       END

```

### Memoria

In [ ]:
class EstadoChat(TypedDict):
    messages: Annotated[list, add_messages]

llm = ChatOllama(
    model=MODELO, 
    temperature=0, 
    num_thread=os.cpu_count(), 
    num_gpu=999, # aplica solo a silicon mac
    model_kwargs={"think": False}
)

def chat(state: EstadoChat) -> dict:
    return {"messages": [llm.invoke(state["messages"])]}

g = StateGraph(EstadoChat)
g.add_node("chat", chat)
g.add_edge(START, "chat")
g.add_edge("chat", END)

app = g.compile(checkpointer=InMemorySaver())
config = {"configurable": {"thread_id": "alumno-001"}}

app.invoke({"messages": [HumanMessage("Me llamo Diego y estudio un IA.")]}, config)
salida = app.invoke({"messages": [HumanMessage("¿Cómo me llamo y qué estudio?")]}, config)

for s in salida["messages"]:
    print(s.type, "->", s.content)

¡Hola! Me llamo Diego y estudio Inteligencia Artificial (IA). ¿Cómo te va? 😊


LangGraph utiliza un Checkpointer simple por defecto para guardar información de la conversación. Para pasos a producción se deberá configurar un PostgresSaver

In [64]:
app = g.compile(checkpointer=InMemorySaver())
config = {"configurable": {"thread_id": "alumno-001"}}

app.invoke({"messages": [HumanMessage("Me llamo Diego y estudio un IA.")]}, config)
salida = app.invoke({"messages": [HumanMessage("¿Cómo me llamo y qué estudio?")]}, config)

app.invoke({"messages": [HumanMessage("Mi segundo nombre es Rafael")]}, config)
salida = app.invoke({"messages": [HumanMessage("¿Cuál es mi primer nombre?¿Cuál es mi segundo nombre?Qué estudio?")]}, config)

for s in salida["messages"]:
    print(s.type, "->", s.content)

human -> Me llamo Diego y estudio un IA.
ai -> Hello Diego! It's great to hear that you're studying AI. How are you doing? What projects or topics are you currently working on in the field of artificial intelligence? 😊
human -> ¿Cómo me llamo y qué estudio?
ai -> ¡Hola! Me llamo Diego y estudio Inteligencia Artificial (IA). ¿Cómo te va? 😊
human -> Mi segundo nombre es Rafael
ai -> ¡Hola Diego! Me llamo Diego Rafael. ¿Cómo te va? 😊
human -> ¿Cuál es mi primer nombre?¿Cuál es mi segundo nombre?Qué estudio?
ai -> ¡Hola! Tu primer nombre es **Diego**, tu segundo nombre es **Rafael** y estudias **Inteligencia Artificial (IA)**. ¿Cómo te va? 😊


# Tarea en clase

Usar el modelo basado en transformers en lugar de ollama:

```
from langchain_huggingface import ChatHuggingFace, HuggingFacePipeline

base = HuggingFacePipeline.from_model_id(
    model_id="Qwen/Qwen3-4B-Instruct",
    task="text-generation",
    pipeline_kwargs={"max_new_tokens": 512, "temperature": 0.0, "do_sample": False},
)
llm = ChatHuggingFace(llm=base)
```

---

En el notebook de la sesión construimos un grafo condicional que clasifica mensajes en tres categorías: `queja`, `pregunta` y `elogio`. En este taller deberás extenderlo.

---

### Instrucciones

1. Copia el grafo clasificador del notebook como punto de partida en una celda nueva.

2. Añade una cuarta categoría: `urgente`. Debes:
- Actualizar el prompt de clasificación para que el LLM pueda devolver esta nueva categoría.
- Crear un nodo `manejar_urgente` que retorne una salida adecuada.
- Actualizar la función `router` para enrutar correctamente a ese nodo.
- Registrar el nodo y sus aristas en el grafo.

3. Prueba el grafo con los siguientes cuatro inputs (uno por categoría esperada):
- *"Llevo tres días esperando mi pedido."*
- *"¿Cómo cancelo mi suscripción?"*
- *"El servicio al cliente fue excelente."*
- *"¡Mi cuenta fue hackeada y no puedo entrar!"*

Imprime el resultado completo de cada invocación.

4. ¿Qué pasa si el LLM devuelve una categoría que la función `router` no conoce? ¿Cómo lo manejarías?